# 🤖 Build a Research Assistant Agent
### From one API call → a single agent → a team of six agents

**Faculty Development Programme · Day 2 · Session 2 (Hands-on I) + Session 3 (Hands-on II)**

**What we will build:** you type a research question → the system finds **real** papers (arXiv, OpenAlex, Semantic Scholar) → it writes a short literature review where **every citation is real and linked**.

> *"As a faculty member, I want to ask a research question and get a short, cited summary of the most relevant recent papers, so I can start a literature review in minutes, not days."*

#### The build ladder (one section per step)

| Session 2 · one agent | Session 3 · a team |
|---|---|
| **1 Call** · plain LLM call | **7 Team** · planner, searchers, screener, reader, writer, critic |
| **2 Tools** · arXiv, OpenAlex, citations | **8 Graph** · LangGraph version with pause/resume |
| **3 Loop** · the agent loop | **9 Ship** · evaluate, cost, deploy |
| **4 Prompt** · system prompt = job description | |
| **5 Memory** · notes that survive across runs | |
| **6 Schema** · validated structured output | |

#### How to use this notebook
- Run cells **top to bottom** (`Shift + Enter`). Every step builds on the previous one.
- Each step is: 📖 **what & why** → 💻 **code** (comments explain each line) → 🧪 **try this**.
- ⚠️ marks a common error and its fix.
- Fell behind? Just run all cells above where the class is (`Runtime → Run before`).

---
# Part 0 · Setup (10 min)

## 0.1 Install the libraries

| Library | What we use it for |
|---|---|
| `openai` | One client that talks to **any** OpenAI-compatible provider (Groq, Gemini, OpenAI, Ollama…) |
| `requests` | Calling the paper APIs over HTTP |
| `feedparser` | arXiv returns Atom XML → this turns it into Python objects |
| `chromadb` | A small vector database for long-term memory |
| `pydantic` | Checks that the model's JSON has the right fields and types |

In [ ]:
!pip install -q openai requests feedparser chromadb pydantic
print("✅ libraries installed")

## 0.2 Add your API key safely (Colab Secrets)

**Never paste a key into a code cell.** Notebooks get shared, and the key goes with them.

1. Get a free key: **Groq** → [console.groq.com/keys](https://console.groq.com/keys) (or Gemini → [aistudio.google.com/apikey](https://aistudio.google.com/apikey))
2. In Colab, click the **🔑 key icon** in the left sidebar → **Add new secret**
3. Name: `GROQ_API_KEY` (or `GEMINI_API_KEY`) → paste the key → switch **Notebook access** ON

📖 **Why "OpenAI-compatible"?** Most providers copy the same API format. So the *same code* works everywhere: we only change `base_url` and `model`.
⚠️ Model names change often. If you get *"model not found"*, check the provider's model list and edit the name below.

In [ ]:
import os

# Every provider below speaks the same "OpenAI-compatible" API.
PROVIDERS = {
    "groq":   {"base_url": "https://api.groq.com/openai/v1",
               "model": "llama-3.3-70b-versatile",      # strong: plans, writes, uses tools
               "small": "llama-3.1-8b-instant",         # small + fast: many cheap calls
               "secret": "GROQ_API_KEY"},
    "gemini": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
               "model": "gemini-2.5-flash", "small": "gemini-2.5-flash",
               "secret": "GEMINI_API_KEY"},
    "openai": {"base_url": None, "model": "gpt-4o-mini", "small": "gpt-4o-mini",
               "secret": "OPENAI_API_KEY"},
}
PROVIDER = "groq"            # 👈 change this one word to switch provider

cfg = PROVIDERS[PROVIDER]
try:                                            # 1st choice: Colab Secrets (🔑 sidebar)
    from google.colab import userdata
    API_KEY = userdata.get(cfg["secret"])
except Exception:                               # outside Colab, or secret missing: ask
    from getpass import getpass
    API_KEY = os.getenv(cfg["secret"]) or getpass(f"Paste your {cfg['secret']}: ")

BASE_URL = cfg["base_url"]
MODEL  = cfg["model"]        # used by the Session 2 agent
STRONG = cfg["model"]        # Session 3: planner, writer, critic (few calls, need quality)
SMALL  = cfg["small"]        # Session 3: screener, reader (many calls, need speed)
print(f"✅ provider = {PROVIDER} · MODEL = {MODEL} · SMALL = {SMALL}")

---
# Session 2 · Building your first agent

# Step 1 · A plain LLM call (6 min)

📖 **What:** send a list of *messages* to the model and get text back. This is the "brain" of every agent.

| role | meaning |
|---|---|
| `system` | who the model is + rules (the job description) |
| `user` | the request |
| `assistant` | what the model replied |
| `tool` | a result from a tool we ran (Step 3) |

In [ ]:
from openai import OpenAI

# One client, configured only by base_url + api_key → same code for every provider
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a research assistant."},                  # who it is
        {"role": "user",   "content": "List 5 recent papers on RAG with year and link."},  # the task
    ],
)
print(resp.choices[0].message.content)   # the reply text
print("\ntokens used:", resp.usage.total_tokens)

🧪 **Try this (2 min):** copy two of those titles into Google Scholar. Do they exist? Are the years and links right?

💡 **The lesson:** a model alone answers **from memory**. It produces titles that *look* right but may not exist (**hallucination**). To be trustworthy, it needs **tools that fetch real records**. That is what turns an LLM into an agent.

### A small helper we will use everywhere: `chat()`

📖 Free tiers sometimes reply **"429: slow down"**. Instead of crashing, this wrapper **waits and retries**, and it **counts tokens** so we can compute cost later (guardrail 6: *timeouts and retries*).

In [ ]:
import time
from openai import RateLimitError, APIConnectionError, BadRequestError

import pandas as pd                          # for nice tables later
from IPython.display import Markdown, display   # to render Markdown answers

USAGE = {"calls": 0, "tokens": 0}           # running total for the cost slide

def chat(**kwargs):
    """Same as client.chat.completions.create, plus retries and token counting."""
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(**kwargs)
            USAGE["calls"] += 1
            USAGE["tokens"] += resp.usage.total_tokens if resp.usage else 0
            return resp
        except (RateLimitError, APIConnectionError) as e:   # 429 or network blip
            wait = 5 * (attempt + 1)                       # wait 5s, 10s, 15s …
            print(f"   ⏳ {type(e).__name__}: waiting {wait}s and retrying…")
            time.sleep(wait)
    raise RuntimeError("Provider still failing after 5 retries.")

print("✅ chat() ready")

---
# Step 2 · Tools (20 min)

📖 A **tool** is a normal Python function the model may *ask* us to run. The model never runs code itself: it says *"please call `search_arxiv` with `query='rag'`"*, and **our** code runs it.

### Six rules for tools an agent can use well
| # | Rule | Example |
|---|---|---|
| 1 | **Clear verb names** | `search_arxiv`, not `tool1` |
| 2 | **Descriptions are prompts** | say *when* to use it and *how* to phrase input |
| 3 | **Few, typed parameters** | limits and types beat free text |
| 4 | **Compact results** | only needed fields; cut long text |
| 5 | **Errors as messages** | return `{"error": "Not found: try broader keywords"}`, don't crash |
| 6 | **Safe by default** | read-only first; confirm before any write |

## Tool 1 · `search_arxiv`

📖 **How it works:**
1. Call the free arXiv API (no key). It returns **Atom XML**.
2. `feedparser` turns XML into Python objects.
3. Keep only 5 fields per paper and cut the abstract to 600 characters (**rule 4**).
4. If the network fails, return an error **message** (**rule 5**), so the agent can recover.
5. arXiv asks for **about 1 request every 3 seconds**, so we wait between calls. The `Lock` makes this safe when several searches run in parallel later (Session 3).

In [ ]:
import json, threading, requests, feedparser

_arxiv_lock, _arxiv_last = threading.Lock(), [0.0]

def search_arxiv(query: str, max_results: int = 5):
    """Search arXiv; return id, title, year, url, abstract for each paper."""
    with _arxiv_lock:                                    # one arXiv call at a time…
        wait = 3 - (time.time() - _arxiv_last[0])        # …at most every 3 seconds
        if wait > 0:
            time.sleep(wait)
        _arxiv_last[0] = time.time()
    try:
        r = requests.get("https://export.arxiv.org/api/query", timeout=30, params={
            "search_query": f"all:{query}",              # search all fields
            "max_results": max_results,
            "sortBy": "relevance"})
        r.raise_for_status()
    except requests.RequestException as e:
        return {"error": f"arXiv unavailable ({e}); try search_openalex"}   # rule 5

    feed = feedparser.parse(r.text)                      # Atom XML → Python
    papers = [{"id":       e.id.split("/abs/")[-1],      # e.g. '2401.01234v1'
               "title":    " ".join(e.title.split()),    # remove line breaks
               "year":     int(e.published[:4]),         # '2024-01-15…' → 2024
               "url":      e.link,
               "abstract": " ".join(e.summary.split())[:600]}   # rule 4: compact
              for e in feed.entries]
    return papers or {"error": "Not found: try broader or different keywords"}

# 🧪 Always test a tool ALONE before giving it to the model (most "agent bugs" are tool bugs)
result = search_arxiv("graph neural networks", max_results=3)
print(json.dumps(result, indent=2)[:1500])

## Tool 2 · `search_openalex`  *(this is lab exercise E1, solved, because Session 3 needs it)*

📖 **Why a second source?** arXiv is mostly CS/physics preprints. **OpenAlex** covers **all disciplines and journals**, needs no key, and gives **citation counts** for free.

📖 **One odd detail:** OpenAlex stores abstracts as an *inverted index* `{"word": [positions]}` (for copyright reasons). We rebuild the text by sorting words by position.

We return the **same keys as `search_arxiv`** (+ `citations`), so the rest of the code doesn't care which source a paper came from.

In [ ]:
def _rebuild_abstract(inverted_index):
    """{'Deep': [0], 'learning': [1]}  →  'Deep learning'"""
    if not inverted_index:
        return ""
    words = sorted((pos, word) for word, positions in inverted_index.items() for pos in positions)
    return " ".join(word for _, word in words)[:600]

def search_openalex(query: str, max_results: int = 5):
    """Search OpenAlex (all fields, journals + citation counts); same keys as search_arxiv."""
    try:
        r = requests.get("https://api.openalex.org/works", timeout=30,
                         params={"search": query, "per-page": max_results})
        r.raise_for_status()
    except requests.RequestException as e:
        return {"error": f"OpenAlex unavailable ({e}); try search_arxiv"}
    papers = [{"id":        w["id"].rsplit("/", 1)[-1],          # 'W2741809807'
               "title":     " ".join(w["title"].split()),
               "year":      w.get("publication_year") or 0,
               "url":       w.get("doi") or w["id"],             # prefer the DOI link
               "abstract":  _rebuild_abstract(w.get("abstract_inverted_index")),
               "citations": w.get("cited_by_count", 0)}
              for w in r.json().get("results", []) if w.get("title")]
    return papers or {"error": "Not found: try broader or different keywords"}

result = search_openalex("graph neural networks", max_results=3)
for p in result if isinstance(result, list) else [result]:
    print(p)

## Tool 3 · `get_citations`

📖 **What:** given **one exact title**, return its citation count, venue and year from **Semantic Scholar**.

⚠️ Without a key, Semantic Scholar shares one rate-limited pool, so a classroom of 40 people will see **HTTP 429**. We handle it two ways:
- the error becomes a **message** the model can read (rule 5)
- we **fall back to OpenAlex** automatically

In [ ]:
def get_citations(title: str) -> dict:
    """Citation count, venue and year for ONE paper title (Semantic Scholar → OpenAlex)."""
    try:
        r = requests.get("https://api.semanticscholar.org/graph/v1/paper/search", timeout=20,
                         params={"query": title, "limit": 1,
                                 "fields": "title,year,citationCount,venue"})
        if r.status_code == 200 and r.json().get("data"):
            return r.json()["data"][0]
    except requests.RequestException:
        pass
    oa = search_openalex(title, max_results=1)            # fallback when S2 says 429
    if isinstance(oa, list):
        return {"title": oa[0]["title"], "year": oa[0]["year"],
                "citationCount": oa[0]["citations"], "source": "openalex"}
    return {"error": "not found"}

print(get_citations("Attention Is All You Need"))

## The `REGISTRY`: name → real code

📖 The model only ever sends us a **name** (a string) and **arguments** (JSON). The registry maps that name to the real Python function. It is also our **allowlist**: if a name isn't here, it doesn't run.

In [ ]:
REGISTRY = {
    "search_arxiv":    search_arxiv,
    "search_openalex": search_openalex,
    "get_citations":   get_citations,
}
print("tools:", list(REGISTRY))

## Describing tools to the model (JSON Schema)

📖 The model **cannot see our Python code**. It only sees this description. So:
- `description` = **the most important line**. It decides *when* the tool is used and *how* arguments are filled.
- `parameters` = standard **JSON Schema**: types, limits, which fields are required.

🧪 **Try later:** change the arXiv description to `"searches stuff"`, re-run the agent, and watch the search queries get worse.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "search_arxiv",
        "description": "Search arXiv for papers. Use 3-6 keywords, not full sentences.",
        "parameters": {"type": "object",
            "properties": {
                "query":       {"type": "string", "description": "3-6 keywords"},
                "max_results": {"type": "integer", "minimum": 1, "maximum": 20}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "search_openalex",
        "description": "Search OpenAlex: all disciplines, journals, with citation counts. "
                       "Use 3-6 keywords. Good when arXiv results are weak or off-field.",
        "parameters": {"type": "object",
            "properties": {
                "query":       {"type": "string", "description": "3-6 keywords"},
                "max_results": {"type": "integer", "minimum": 1, "maximum": 20}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "get_citations",
        "description": "Citation count, venue and year for ONE exact paper title.",
        "parameters": {"type": "object",
            "properties": {"title": {"type": "string"}},
            "required": ["title"]}}},
]
print(f"✅ {len(TOOLS)} tool schemas")

---
# Step 3 · The agent loop (25 min)

📖 **An agent = an LLM in a loop with tools.** The model decides the next step; our code carries it out and reports back.

```
  ┌────────────┐     ┌──────────────────┐   no    ┌───────────────┐
  │  messages  │────►│ LLM call (TOOLS) │───────► │ return answer │
  └────────────┘     └────────┬─────────┘  tool   └───────────────┘
        ▲                     │ yes        calls?
        │            ┌────────▼─────────┐
        └────────────│ run REGISTRY[name]│  append result as role="tool"
                     └──────────────────┘
   Also stop when: max_steps reached · token budget spent · same call repeats
```

The **only decision** in the loop: *did the model ask for tools?*
- **Yes** → run them, append the results, go round again
- **No** → that message is the final answer

### A tiny helper: `assistant_message()`

📖 We must put the model's reply back into `messages` so it remembers what it asked for. Some providers reject extra fields, so we keep only the 3 standard ones: `role`, `content`, `tool_calls`.

In [ ]:
def assistant_message(msg) -> dict:
    """Model reply → plain dict with only the portable fields."""
    d = msg.model_dump(exclude_none=True)
    d = {k: v for k, v in d.items() if k in ("role", "content", "tool_calls")}
    d.setdefault("content", "")
    return d

def short(result) -> str:
    """One-line summary of a tool result, for the trace."""
    if isinstance(result, list):
        return f"{len(result)} items"
    if isinstance(result, dict) and "error" in result:
        return f"⚠️ {result['error']}"
    return str(result)[:80]

### The whole agent in ~20 lines (version 1)

Map each part to the flowchart:
| Code | Flowchart box |
|---|---|
| `messages = [...]` | Start |
| `chat(... tools=TOOLS)` | LLM call |
| `if not msg.tool_calls` | the diamond: *tool calls?* |
| inner `for call in ...` | run tool + append result |
| last `return` | safety exit (step limit) |

In [ ]:
SYSTEM_PROMPT = "You are a research assistant. Use the tools to find real papers."   # better one in Step 4

def run_agent(goal: str, max_steps: int = 8) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": goal}]
    for step in range(max_steps):
        resp = chat(model=MODEL, messages=messages, tools=TOOLS)   # model sees the tool list
        msg = resp.choices[0].message
        messages.append(assistant_message(msg))                   # remember what it said

        if not msg.tool_calls:                                     # 1. no tools → final answer
            return msg.content

        for call in msg.tool_calls:                                # 2. run each requested tool
            args = json.loads(call.function.arguments)             #    arguments arrive as JSON text
            result = REGISTRY[call.function.name](**args)          #    name → real function
            print(f"[{step}] 🔧 {call.function.name}({args}) → {short(result)}")   # the trace
            messages.append({"role": "tool",
                             "tool_call_id": call.id,              #    links result to the request
                             "content": json.dumps(result)[:4000]})   # cut so context won't flood
    return "Stopped: step limit reached."                          # 3. safety exit

answer = run_agent("Recent papers evaluating hallucination in RAG")
display(Markdown(answer))

### 🔍 Read the trace (this matters more than the answer)

A typical run looks like this (*yours will differ, because the model chooses*):

| Step | The model decides… | …and the trace shows |
|---|---|---|
| 0 | search with focused keywords | `search_arxiv(query="RAG hallucination evaluation")` |
| 1 | results too broad → **narrow the query** | `search_arxiv(query="RAG faithfulness benchmark")` |
| 2 | check impact of top papers | `get_citations(...)` × 3 **in one step** (parallel calls) |
| 3 | enough evidence | no tool calls → final answer |

💡 **Discuss:** query refinement was the agent's **own idea**. Agents are **non-deterministic**, so run it twice and compare. We judge an agent over many runs, not one.

---
# Step 4 · The system prompt is the job description (4 min)

📖 A good agent prompt has **4 parts**:

| Part | In our prompt |
|---|---|
| **Role and goal** | research assistant for faculty; find + summarise papers |
| **Rules** | grounding (*only cite tool results*), recency, citation format |
| **Stopping rule** | at most 3 searches |
| **Output format** | what "done" looks like |

🧪 **Try:** change the first line to *"…for **civil engineering** faculty"* and see how the search queries change.

In [ ]:
SYSTEM_PROMPT = """
You are a research assistant for university faculty.
Goal: find and summarise relevant recent papers.
Rules:
1. Use tools for every claim about a paper. Never invent
   titles, authors, years or numbers. Cite only papers a tool returned.
2. Prefer papers from the last 3 years unless asked.
3. Cite each paper as [Title](url).
4. If results are weak, rewrite the query and search
   again, at most 3 searches.
Output: key papers (1 line each), main findings,
open gaps, suggested next reading.
"""
answer = run_agent("Recent papers evaluating hallucination in RAG")
display(Markdown(answer))

---
# Step 5 · Long-term memory (8 min)

| Memory type | Where | Lifetime |
|---|---|---|
| **Short-term** | the `messages` list | gone when the run ends |
| **Long-term** | a vector database (Chroma) | next run, next week |

📖 **How a vector database works:** each note is turned into an **embedding** (a list of numbers that captures meaning). `recall_notes("hallucination")` finds notes with *similar meaning*, even if the exact words differ.

⚠️ The first call downloads a small embedding model (~80 MB, once). If Chroma fails, the cell falls back to a simple keyword memory so the class can continue.
⚠️ Colab deletes files when the runtime resets. For real persistence, mount Google Drive and set `path` to a Drive folder.

In [ ]:
import re
try:
    import chromadb
    db = chromadb.PersistentClient(path="research_memory")       # saved to disk
    notes = db.get_or_create_collection("paper_notes")
    notes.upsert(ids=["_warmup"], documents=["warm-up note"])   # triggers the one-time model download
    notes.delete(ids=["_warmup"])
    USE_CHROMA = True
except Exception as e:
    print("⚠️ Chroma unavailable, using simple keyword memory instead:", str(e)[:100])
    USE_CHROMA, NOTES = False, {}

def save_note(paper_id: str, note: str) -> str:
    """Store a short note about one paper (same id → overwritten, not duplicated)."""
    if USE_CHROMA:
        notes.upsert(ids=[paper_id], documents=[note])
    else:
        NOTES[paper_id] = note
    return "saved"

def recall_notes(query: str, k: int = 3) -> list:
    """Return up to k saved notes most related to the query."""
    if USE_CHROMA:
        n = notes.count()
        if n == 0:
            return ["(no notes saved yet)"]
        return notes.query(query_texts=[query], n_results=min(k, n))["documents"][0]
    words = set(re.findall(r"\w+", query.lower()))               # fallback: count shared words
    ranked = sorted(NOTES.values(), key=lambda t: -len(words & set(re.findall(r"\w+", t.lower()))))
    return ranked[:k] or ["(no notes saved yet)"]

# give the two new tools to the agent: real code + description
REGISTRY |= {"save_note": save_note, "recall_notes": recall_notes}
TOOLS += [
    {"type": "function", "function": {
        "name": "recall_notes",
        "description": "Recall notes saved in earlier runs. Call this BEFORE searching.",
        "parameters": {"type": "object",
            "properties": {"query": {"type": "string"}, "k": {"type": "integer", "minimum": 1, "maximum": 10}},
            "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "save_note",
        "description": "Save a two-line note about one paper you used, for future runs.",
        "parameters": {"type": "object",
            "properties": {"paper_id": {"type": "string", "description": "id from the search result"},
                           "note": {"type": "string", "description": "Title (year): key finding"}},
            "required": ["paper_id", "note"]}}},
]
SYSTEM_PROMPT += """
Memory: before searching, call recall_notes to check what you already know.
After choosing the key papers, save a two-line note for each with save_note.
"""
print("memory:", "Chroma" if USE_CHROMA else "keyword fallback", "· tools:", list(REGISTRY))

---
# Guardrails · seven limits every lab agent needs (4 min)

📖 Version 1 of the loop trusts the model completely. Version 2 adds cheap safety checks that turn a demo into something you can leave running for students:

| # | Guardrail | Why |
|---|---|---|
| 1 | `max_steps` | stops infinite loops |
| 2 | token budget | caps cost per run |
| 3 | truncate tool output | context window doesn't flood |
| 4 | validate arguments | bad JSON → error message, not a crash |
| 5 | allowlist of tools | models sometimes **invent** tool names |
| 6 | timeouts and retries | already inside `chat()` and `requests` |
| 7 | log every step | you debug from the trace |
| + | repeat-call block | stops "search the same thing again and again" |
| + | force the first tool call | fixes "never calls a tool" |

In [ ]:
TOKEN_BUDGET = 60_000

def run_agent(goal: str, max_steps: int = 8) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": goal}]
    used, seen = 0, set()                                    # tokens spent · calls already made
    for step in range(max_steps):                            # 🛡️1 step limit
        kwargs = dict(model=MODEL, messages=messages, tools=TOOLS)
        if step == 0:
            kwargs["tool_choice"] = "required"               # 🛡️ force a tool on step 0
        try:
            resp = chat(**kwargs)
        except BadRequestError as e:                         # provider rejected this request
            print("   ⚠️ provider error, retrying without tool_choice:", str(e)[:100])
            kwargs.pop("tool_choice", None)
            resp = chat(**kwargs)

        used += resp.usage.total_tokens if resp.usage else 0
        if used > TOKEN_BUDGET:                              # 🛡️2 token budget
            return "Stopped: token budget reached."

        msg = resp.choices[0].message
        messages.append(assistant_message(msg))
        if not msg.tool_calls:
            print(f"[{step}] ✅ final answer · {used} tokens")
            return msg.content

        for call in msg.tool_calls:
            name, raw_args = call.function.name, call.function.arguments or "{}"
            if name not in REGISTRY:                         # 🛡️5 allowlist
                result = {"error": f"unknown tool {name}; use one of {list(REGISTRY)}"}
            elif (name, raw_args) in seen:                   # 🛡️ repeated call
                result = {"error": "already tried this exact call; change the query or answer now"}
            else:
                try:
                    result = REGISTRY[name](**json.loads(raw_args))   # 🛡️4 bad args → caught below
                except Exception as e:
                    result = {"error": f"{type(e).__name__}: {e}"}
            seen.add((name, raw_args))
            print(f"[{step}] 🔧 {name}({raw_args}) → {short(result)}")   # 🛡️7 log
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)[:4000]})     # 🛡️3 truncate
    return "Stopped: step limit reached."

print("✅ run_agent v2 (with guardrails) ready")

### 🧪 Prove the memory works: run twice on related questions

- **Run 1:** `recall_notes` finds nothing → the agent searches → saves notes
- **Run 2:** `recall_notes` returns notes from run 1 → the agent can reuse them (watch the trace)

In [ ]:
print("=========== RUN 1 ===========")
display(Markdown(run_agent("Recent papers evaluating hallucination in RAG")))

In [ ]:
print("=========== RUN 2 (related question) ===========")
display(Markdown(run_agent("How do researchers measure faithfulness of retrieval-augmented LLMs?")))
print("\nWhat memory now holds:", recall_notes("hallucination", k=5))

---
# Step 6 · Structured output you can trust (7 min)

📖 **Problem:** free text is hard to compare. For a review table we need **the same fields for every paper**.

📖 **Solution, in 3 parts:**
1. **JSON mode:** `response_format={"type": "json_object"}` makes the model reply with JSON.
2. **Pydantic** checks it: right fields, right types, `relevance` between 1 and 5. A bad field raises an error **we can retry on**.
3. **Metadata from the API wins:** title, year and URL are copied from the real record, never trusted from the model.

### First, `call_json()`: a helper reused by every Session 3 agent
JSON mode in → Python dict out. It also cleans up ```` ```json ```` fences, and falls back if a provider doesn't support JSON mode.

In [ ]:
def parse_json(text: str) -> dict:
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", (text or "").strip())   # strip ```json fences
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.search(r"\{.*\}", text, re.S).group(0))       # first {...} block

def call_json(system: str, user: str, model: str = None) -> dict:
    """System + user prompt → Python dict (JSON mode)."""
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    try:
        r = chat(model=model or MODEL, messages=msgs, response_format={"type": "json_object"})
    except BadRequestError:                     # provider/model without JSON mode
        r = chat(model=model or MODEL, messages=msgs)
    return parse_json(r.choices[0].message.content)

def call_text(system: str, user: str, model: str = None) -> str:
    """Same as call_json, but returns plain text (used by the writer)."""
    r = chat(model=model or MODEL, messages=[{"role": "system", "content": system},
                                             {"role": "user", "content": user}])
    return r.choices[0].message.content or ""

print("✅ call_json / call_text ready")

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class PaperSummary(BaseModel):
    title: str
    year: int
    url: str
    problem: str = Field(description="question the paper tackles")
    method: str
    key_result: str
    limitation: str
    relevance: int = Field(ge=1, le=5, description="relevance to the research question, 1-5")

READER = ("Summarise ONLY from the text given. If a field is not stated, write 'not reported'. "
          "Return JSON matching this schema: " + json.dumps(PaperSummary.model_json_schema()))

def summarise_paper(paper: dict, question: str = "", model: str = None) -> PaperSummary:
    """One paper's abstract → a validated PaperSummary."""
    user = f"Research question: {question}\n\nTitle: {paper['title']}\n\nAbstract: {paper['abstract']}"
    meta = {"title": paper["title"], "year": int(paper["year"] or 0), "url": paper["url"]}
    for attempt in range(2):
        data = call_json(READER, user, model=model)
        try:
            return PaperSummary(**{**data, **meta})          # 👈 API metadata overwrites the model's
        except ValidationError as e:
            print("   ⚠️ validation failed, retrying with the error message")
            user += f"\n\nYour previous JSON failed validation: {e}. Fix it."
    return PaperSummary(**meta, problem="not reported", method="not reported",
                        key_result="not reported", limitation="not reported", relevance=1)

# 🧪 test it on one real paper
hits = search_arxiv("retrieval augmented generation hallucination", 1)
if isinstance(hits, dict):
    print(hits)                                              # error message: re-run in a minute
else:
    s = summarise_paper(hits[0], "hallucination in RAG")
    print(s.model_dump_json(indent=2))

---
# 🩺 Debugging clinic: five failures you will see today

| Symptom | Likely cause | Fix |
|---|---|---|
| Never calls a tool | model lacks tool support | use a tool-capable model; force the first call (we do) |
| Invents papers | answers from memory | rule: cite only tool results; check URLs |
| Repeats a search | no stopping rule | step limit; repeat-call block (we do) |
| Bad arguments | vague tool description | sharper description, limits, examples |
| Context too long | raw results pile up | truncate; summarise; move notes to memory |
| ⚠️ Groq `tool_use_failed` (400) | model wrote a malformed tool call | just re-run the cell; or try another model |
| ⚠️ `429` / rate limit | free tier limit | `chat()` waits and retries automatically |

> **Always debug from the trace:** read what the model *saw* before blaming what it *did*.

---
# ✍️ Your turn · Lab exercises (15 min)

| | Exercise | Level |
|---|---|---|
| **E1** | Third tool `search_openalex` | ✅ already done above |
| **E2** | Make the agent call `recall_notes` **before any search**, and prove it with the trace | Easy |
| **E3** | Add a `year_from` filter to `search_arxiv` and update its schema | Medium |
| **E4** | Return a JSON list of 5 validated `PaperSummary` objects for **your own** topic | Stretch |

Solutions for E3 and E4 are below. **Try first, then run them.**

In [ ]:
# ---- E3 solution: filter by year in Python (more reliable than arXiv's query syntax)
def search_arxiv_since(query: str, year_from: int, max_results: int = 5):
    res = search_arxiv(query, max_results * 3)               # fetch extra, then filter
    if isinstance(res, dict):
        return res                                           # pass the error message through
    res = [p for p in res if p["year"] >= int(year_from)][:max_results]
    return res or {"error": f"Nothing since {year_from}: try broader keywords or an earlier year"}

res = search_arxiv_since("large language models education", year_from=2024, max_results=3)
if isinstance(res, dict):
    print(res)                                               # an error message
else:
    for p in res:
        print(p["year"], p["title"])

In [ ]:
# ---- E4 solution: a validated reading list  (👈 change MY_TOPIC to your own field)
MY_TOPIC = "deep learning cardiac risk prediction"
hits = search_arxiv(MY_TOPIC, 5)
reading_list = [summarise_paper(p, MY_TOPIC).model_dump() for p in hits] if isinstance(hits, list) else []
pd.DataFrame(reading_list)[["year", "title", "key_result", "relevance"]] if reading_list else hits

---
# ✅ Checkpoint: one agent works. Where does it strain?

| You built | Its limits |
|---|---|
| tools for arXiv, OpenAlex, Semantic Scholar | **one prompt doing five jobs** (plan, search, rank, read, write) |
| a bounded agent loop with a trace | **context fills up** on broad questions |
| persistent notes + validated summaries | **nobody checks its work** before you see it |

➡️ **Session 3:** split it into a **team**, and add a **critic**.

---
# Session 3 · The Research Assistant: a team of six agents

## Why split the agent?
We split because **one prompt + one context was overloaded**, not because multi-agent is fashionable. Each agent is a stage a human researcher already does:

```
 INTERFACE      faculty user ──► (Streamlit UI · FastAPI)
 ORCHESTRATION  orchestrator · shared ResearchState · human checkpoint · revise loop
 AGENTS         Planner → Searchers×3 → Screener → [HUMAN] → Reader → Writer ⇄ Critic
 TOOLS          arXiv · OpenAlex · Semantic Scholar · Chroma notes
```

## Nine stages, question → report
`1 Clarify` → `2 Plan` → `3 Search (parallel)` → `4 Screen` → **`5 Approve (human)`** → `6 Read` → `7 Write` ⇄ `8 Verify` → `9 Deliver`
*(stages 7–8 repeat until the critic passes the draft, at most 2 revisions)*

## Agent roster: least privilege + model tiering
| Agent | Job | Tools | Output | Model |
|---|---|---|---|---|
| Planner | split the question | none | sub-questions | strong |
| Searcher | find candidates | arXiv, OpenAlex | paper list | (plain code) |
| Screener | dedupe + rank | get_citations | top-k shortlist | small |
| Reader | extract key facts | save_note | PaperSummary | small |
| Writer | synthesise by theme | none | Markdown draft | strong |
| Critic | verify claims + citations | code check | pass + issues | strong |

💡 Many cheap calls → **small** model. Few judgement calls → **strong** model. The writer **cannot search**, so it can't pull in papers the human didn't approve.

## The shared state: our blackboard

📖 Design the **state before the agents.** Every agent:
1. **reads** the whole state
2. **returns only the keys it changed** (a small dict)
3. the orchestrator **merges** it: `state |= update`

This is exactly LangGraph's convention, so the same functions plug into a graph later. The state is also an **audit trail** of how each paper was chosen.

In [ ]:
from typing import TypedDict

class ResearchState(TypedDict, total=False):
    question: str           # from the user
    plan: list              # sub-questions + keyword queries        ← planner
    candidates: list        # raw search hits                          ← searchers
    shortlist: list         # ranked, then human-approved              ← screener + human
    summaries: list         # one PaperSummary per paper               ← reader
    draft: str              # Markdown literature review               ← writer
    review: dict            # {"passed": bool, "issues": [...]}        ← critic
    revisions: int          # how many writer passes so far

QUESTION = "LLM hallucination in medical question answering"     # 👈 put your own question here
state: ResearchState = {"question": QUESTION, "revisions": 0}
print(state)

## Agent 1 of 6 · Planner: question → sub-questions

📖 **Pattern: prompt chaining.** A broad question gives bad searches. The planner splits it into 3–5 sub-questions, each with 2–3 short keyword queries. No tools: it only thinks.

🧪 **Read the sub-questions aloud: is anything important missing?** That is the first place a human adds value.

In [ ]:
PLANNER = ("Split the research question into 3-5 sub-questions that together cover it. "
           "For each, give 2-3 keyword queries (3-6 words each). Return JSON: "
           '{"subquestions": [{"q": str, "queries": [str]}]}')

def planner(state):
    out = call_json(PLANNER, state["question"], model=STRONG)
    return {"plan": out["subquestions"][:5]}          # only the key it changed

state |= planner(state)                               # merge the update into the state
for s in state["plan"]:
    print("❓", s["q"], "\n    🔎", " | ".join(s["queries"]))

## Agent 2 of 6 · Searchers: fan out, then merge

📖 **Pattern: parallelisation.** One searcher per sub-question, running **at the same time** (`ThreadPoolExecutor`). Each runs every query on arXiv **and** OpenAlex, then all results are merged into one list.

📖 These searchers are **plain functions** (a *workflow step*): cheap and predictable. Upgrade path: replace `search_one` with `run_agent(sub["q"])` so each searcher becomes a full agent that refines its own queries.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def search_one(sub: dict) -> list:
    hits = []
    for q in sub["queries"][:3]:
        for tool in (search_arxiv, search_openalex):
            r = tool(q, max_results=8)
            if isinstance(r, list):                   # skip error messages, keep going
                hits += r
    return hits

def searchers(state):
    with ThreadPoolExecutor(max_workers=3) as pool:   # 3 searchers in parallel
        batches = list(pool.map(search_one, state["plan"]))
    return {"candidates": [paper for batch in batches for paper in batch]}

t0 = time.time()
state |= searchers(state)
print(f"✅ {len(state['candidates'])} raw hits in {time.time() - t0:.0f}s")

## Agent 3 of 6 · Screener: many hits → a shortlist

📖 **Three steps:**
1. **De-duplicate** by normalised title (lowercase, no punctuation). The same paper often comes from both sources.
2. **Score** every paper with an explicit, documented formula:

$$\text{score} = 0.5 \cdot \text{relevance} + 0.2 \cdot \text{recency} + 0.3 \cdot \text{citations}$$

| Part | How we compute it (all scaled 0–1) |
|---|---|
| relevance | small LLM grades each paper 1–5 (in batches of 25 = few calls) |
| recency | 1.0 if ≤ 3 years old, then −0.15 per extra year |
| citations | `log(1 + c) / log(1 + max_c)`. The log stops one famous paper from dominating |

3. Keep the **top k = 12**.

💡 The weights are a **starting point to discuss**: a historian may weight citations lower; a fast AI subfield may weight recency higher. Writing the formula down is good research practice, like the screening stage of a systematic review.

In [ ]:
import math
from datetime import date

GRADER = ("Grade how relevant each paper is to the research question, from 1 (off-topic) to 5 (core). "
          'Return JSON {"scores": [int, ...]} with one score per paper, in the same order.')

def norm(title: str) -> str:
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", title.lower()).split())

def recency(year) -> float:
    age = date.today().year - int(year or 0)
    return 1.0 if age <= 3 else max(0.0, 1 - 0.15 * (age - 3))

def relevance_scores(papers: list, question: str) -> list:
    scores = []
    for i in range(0, len(papers), 25):                              # batches of 25 papers
        chunk = papers[i:i + 25]
        listing = "\n".join(f"{j}. {p['title']} :: {p['abstract'][:200]}" for j, p in enumerate(chunk))
        grades = call_json(GRADER, f"QUESTION: {question}\n\nPAPERS:\n{listing}", model=SMALL).get("scores", [])
        if len(grades) != len(chunk):
            grades = [3] * len(chunk)                                # model miscounted → neutral grade
        scores += [(max(1, min(5, int(g))) - 1) / 4 for g in grades] # 1..5 → 0..1
    return scores

def screener(state, k=12):
    # 1. de-duplicate (keep the non-empty fields from both copies)
    uniq = {}
    for p in state["candidates"]:
        old = uniq.get(norm(p["title"]), {})
        uniq[norm(p["title"])] = {**p, **{key: v for key, v in old.items() if v}}
    papers = list(uniq.values())

    # 2a. relevance (LLM) — then pre-sort so we only look up citations for promising papers
    for p, r in zip(papers, relevance_scores(papers, state["question"])):
        p["rel"] = r
    papers.sort(key=lambda p: 0.5 * p["rel"] + 0.2 * recency(p["year"]), reverse=True)
    for p in papers[:2 * k]:
        if p.get("citations") is None:                               # arXiv hits have no count
            c = get_citations(p["title"])
            same = "error" not in c and norm(c.get("title") or "")[:40] == norm(p["title"])[:40]
            p["citations"] = (c.get("citationCount") or 0) if same else 0

    # 2b. final score
    max_c = max([p.get("citations") or 0 for p in papers] + [1])
    for p in papers:
        cite = math.log1p(p.get("citations") or 0) / math.log1p(max_c)
        p["score"] = round(0.5 * p["rel"] + 0.2 * recency(p["year"]) + 0.3 * cite, 3)

    # 3. top k
    return {"shortlist": sorted(papers, key=lambda p: p["score"], reverse=True)[:k]}

state |= screener(state)
print(f"{len(state['candidates'])} raw hits → {len(state['shortlist'])} shortlisted")
pd.DataFrame(state["shortlist"])[["score", "rel", "year", "citations", "title"]]

## 🙋 Human checkpoint · approve the shortlist

📖 This is **required**. The faculty member removes off-topic papers **before** the most expensive step (reading every paper). Human time saves machine cost, and it keeps the scholar in charge (academic integrity).

Type the numbers of papers to **drop** (e.g. `3,7`), press Enter to keep all, or `q` to stop.

In [ ]:
AUTO_APPROVE = False          # set True to skip the question (e.g. when running a test set)

def human_approves(state):
    """Show the shortlist; return the approved papers, or None to stop."""
    for i, p in enumerate(state["shortlist"], 1):
        print(f"{i:>2}. [{p['score']}] ({p['year']}, {p.get('citations', 0)} cites) {p['title']}")
    if AUTO_APPROVE:
        return state["shortlist"]
    ans = input("\nDrop which? (e.g. 3,7 · Enter = keep all · q = stop): ").strip().lower()
    if ans == "q":
        return None
    drop = {int(x) for x in re.findall(r"\d+", ans)}
    return [p for i, p in enumerate(state["shortlist"], 1) if i not in drop]

approved = human_approves(state)
if approved:
    state["shortlist"] = approved
print(f"\n✅ {len(state['shortlist'])} papers approved for reading")

## Agent 4 of 6 · Reader: facts from the source, not the model

📖 We reuse `summarise_paper` from Step 6. Two **anti-hallucination techniques** are already inside it:
1. The prompt gives the model an honest way out: **"not reported"** beats a guess.
2. **Title, year and link come from the API record**, so even if the model garbles a title, the stored one is right.

It also **saves a note** per paper → the memory from Session 2 now fills up automatically.

In [ ]:
def reader(state):
    out = []
    for i, p in enumerate(state["shortlist"], 1):
        print(f"  📖 {i}/{len(state['shortlist'])} {p['title'][:80]}")
        s = summarise_paper(p, state["question"], model=SMALL)       # small model: many calls
        save_note(p["id"], f"{s.title} ({s.year}): {s.key_result}")  # long-term memory
        out.append(s.model_dump())
    return {"summaries": out}

state |= reader(state)
pd.DataFrame(state["summaries"])[["year", "title", "method", "key_result", "relevance"]]

## Agent 5 of 6 · Writer: synthesise by theme, cite by number

📖 **Two key design choices:**
1. **Group by THEME, not by paper.** That is what turns a list of summaries into a *literature review*.
2. **Numbered citations `[1]…[n]`.** The writer can only cite papers in the list we give it. The reference list is then **generated in code** from real records, never written by the model.

If the critic found problems (`FIX`), the writer gets them on the next pass.

In [ ]:
WRITER = ("Write a literature review in Markdown. Group findings by THEME, not by paper. "
          "Cite as [n] using ONLY the numbered papers given. Note disagreements and open gaps. "
          "Sections: ## 1. Scope and method, ## 2. Themes, ## 3. Comparison table "
          "(paper [n] | method | key result | limitation), ## 4. Gaps and future work. "
          "Do NOT write a references list. If FIX lists issues, fix every one of them.")

def writer(state):
    papers = "\n".join(f"[{i+1}] {json.dumps(s)}" for i, s in enumerate(state["summaries"]))
    fix = state.get("review", {}).get("issues", [])                # critic feedback (empty at first)
    draft = call_text(WRITER, f"Q: {state['question']}\n\nPAPERS:\n{papers}\n\nFIX: {fix}", model=STRONG)
    return {"draft": draft, "revisions": state.get("revisions", 0) + 1}

def references(state) -> str:
    """Built in CODE from the real records → every reference exists and is linked."""
    return "\n".join(f"[{i+1}] {s['title']} ({s['year']}). {s['url']}"
                     for i, s in enumerate(state["summaries"]))

state |= writer(state)
display(Markdown(state["draft"][:3000] + "\n\n…"))

## Agent 6 of 6 · Critic: code checks first, then an LLM judge

📖 **Pattern: evaluator–optimiser** (writer ⇄ critic). Verification is **layered**:

| Layer | Checks | Certainty |
|---|---|---|
| **1. Code** | does every `[n]` point to a real paper in the list? | 100% certain, free |
| **2. LLM judge** | is each claim actually supported by its source? overstatements? | judgement |

The **code check can veto** the LLM: if any citation is invalid, the draft fails, whatever the judge says.

`after_critic` decides the next edge: **end** if passed or after 3 writer passes (= at most 2 revisions), otherwise **revise**.
💡 Ideally the critic uses a *different* model from the writer, so it isn't marking its own homework.

In [ ]:
CRITIC = ("Strict reviewer: check every claim is supported by its cited [n]; "
          "flag uncited claims and overstatements. Be concrete and brief. "
          'Return JSON {"passed": bool, "issues": [str]}.')

def cited_ids(text: str) -> list:
    return [int(x) for x in re.findall(r"\[(\d+)\]", text)]          # every [n] in the draft

def critic(state):
    n_max = len(state["summaries"])
    bad = sorted({f"[{n}] is not in the paper list" for n in cited_ids(state["draft"])
                  if not 1 <= n <= n_max})                           # layer 1: deterministic
    v = call_json(CRITIC, f"DRAFT:\n{state['draft']}\n\nPAPERS:\n{json.dumps(state['summaries'])}",
                  model=STRONG)                                      # layer 2: LLM judge
    return {"review": {"passed": bool(v.get("passed")) and not bad,  # code check can veto
                       "issues": bad + [str(i) for i in v.get("issues", [])]}}

def after_critic(state) -> str:
    done = state["review"]["passed"] or state["revisions"] >= 3
    return "end" if done else "revise"

state |= critic(state)
print("passed:", state["review"]["passed"])
for issue in state["review"]["issues"]:
    print(" •", issue)
print("next edge →", after_critic(state))

## 🎼 The orchestrator: wiring the team together

📖 Now we put all six agents in one function. Point at each line and name the pattern from the morning:

| Lines | Pattern |
|---|---|
| `planner → searchers → screener` | **prompt chaining** |
| inside `searchers` | **parallelisation** |
| `human_approves` | **human checkpoint** |
| `while True: writer → critic` | **evaluator–optimiser** |

⚠️ **Limitation of plain Python:** if it crashes during reading, everything is lost. A graph framework (next section) adds **checkpoints and resume**.

In [ ]:
def research(question: str) -> ResearchState:
    state: ResearchState = {"question": question, "revisions": 0}
    for step in (planner, searchers, screener):          # prompt chaining
        print(f"▶ {step.__name__}")
        state |= step(state)
    approved = human_approves(state)                     # 🙋 human checkpoint
    if not approved:
        return state
    state["shortlist"] = approved
    print("▶ reader");  state |= reader(state)
    while True:                                          # evaluator–optimiser loop
        print(f"▶ writer (pass {state['revisions'] + 1})");  state |= writer(state)
        print("▶ critic");                                    state |= critic(state)
        if after_critic(state) == "end":
            return state

def final_report(state) -> str:
    rv = state["review"]
    status = "✅ passed" if rv["passed"] else f"⚠️ not passed after {state['revisions']} drafts: {rv['issues']}"
    return (f"# {state['question']}\n\n{state['draft']}\n\n## References\n\n{references(state)}"
            f"\n\n---\n*Critic review: {status}*")

result = research("Deep learning for early detection of diabetic retinopathy")   # 👈 try your own
display(Markdown(final_report(result)))

In [ ]:
# 💾 Save the report + the full state (= audit trail of how every paper was chosen)
with open("literature_review.md", "w") as f:
    f.write(final_report(result))
with open("research_state.json", "w") as f:
    json.dump(result, f, indent=1, default=str)
print("saved literature_review.md and research_state.json  (Colab: 📁 Files panel → download)")

---
# Step 8 · The same graph in LangGraph (instructor demo)

📖 **Not one agent function changes.** Because each returns a *partial state update*, they plug straight in as **nodes**. What we gain:

| Gain | How |
|---|---|
| pause and resume | `interrupt_before=["read"]` = the human checkpoint |
| crash recovery | the checkpointer saves state after every node |
| drawable graph | `draw_mermaid()` |

`thread_id` means a faculty member can approve the shortlist **hours later**, and the run resumes exactly where it stopped.

In [ ]:
!pip install -q langgraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

g = StateGraph(ResearchState)
for name, fn in [("plan", planner), ("search", searchers), ("screen", screener),
                 ("read", reader), ("write", writer), ("critique", critic)]:
    g.add_node(name, fn)                                  # same functions as above
g.add_edge(START, "plan");      g.add_edge("plan", "search")
g.add_edge("search", "screen"); g.add_edge("screen", "read")
g.add_edge("read", "write");    g.add_edge("write", "critique")
g.add_conditional_edges("critique", after_critic, {"revise": "write", "end": END})   # the loop
app = g.compile(checkpointer=MemorySaver(), interrupt_before=["read"])               # 🙋 pause

print(app.get_graph().draw_mermaid())

In [ ]:
cfg = {"configurable": {"thread_id": "review-042"}}
app.invoke({"question": "Federated learning for medical imaging", "revisions": 0}, cfg)   # runs, then PAUSES

paused = app.get_state(cfg)
print("⏸ paused before:", paused.next)
shortlist = paused.values["shortlist"]
for i, p in enumerate(shortlist, 1):
    print(f"{i:>2}. {p['title']}")

approved = shortlist[:8]                                  # (in a UI, the human ticks boxes here)
app.update_state(cfg, {"shortlist": approved})           # human edits the state
app.invoke(None, cfg)                                    # ▶ resume from the checkpoint
display(Markdown(final_report(app.get_state(cfg).values)))

---
# Connect · MCP: publish our tools once, use them anywhere

📖 **MCP (Model Context Protocol)** turns a typed, documented Python function into a tool that **any** MCP client can use: Claude Desktop, IDEs, a colleague's agent framework. The schema comes from the type hints and docstring, so no hand-written JSON.

*(Run this as a script on a laptop, not in Colab; the shared repo has `mcp_server.py`.)*

```python
# pip install mcp
from mcp.server.fastmcp import FastMCP          # mcp 2.x: from mcp.server.mcpserver import MCPServer

mcp = FastMCP("scholar-tools")

@mcp.tool()
def search_arxiv(query: str, max_results: int = 5) -> list[dict]:
    """Search arXiv for papers. Use 3-6 keywords."""
    ...  # same body as Step 2

if __name__ == "__main__":
    mcp.run()
```
**MCP below** (agent ↔ tools) · **A2A above** (agent ↔ agent, e.g. other departments' assistants).

---
# Step 9 · Evaluate: how do we know it is any good?

| Metric | Question it answers | How to measure |
|---|---|---|
| **Citation validity** | does every reference exist and match? | automatic, **target 100%** |
| Retrieval recall | did it find what an expert would? | expert's key-paper list per question |
| Faithfulness | is each claim backed by its source? | LLM judge + human spot-checks |
| Coverage | are all sub-questions answered? | rubric 1–5 |
| Usefulness | would faculty start a review from it? | faculty rating 1–5 |
| Cost, latency | what does one report cost? | logged tokens, ₹, minutes |

📖 Agents are non-deterministic, so **one good demo proves little**. Build a test set of 10–20 questions from your department and re-run it after every change.

In [ ]:
def citation_validity(state) -> float:
    """% of [n] citations that point to a real paper with a URL. Should be 100% by design."""
    ids = cited_ids(state.get("draft", ""))
    s = state.get("summaries", [])
    ok = [n for n in ids if 1 <= n <= len(s) and s[n - 1]["url"].startswith("http")]
    return round(100 * len(ok) / len(ids), 1) if ids else 0.0

print(f"Citation validity: {citation_validity(result)}%")
print(f"Critic passed: {result['review']['passed']} after {result['revisions']} draft(s)")
print(f"Sub-questions planned: {len(result['plan'])} · papers read: {len(result['summaries'])}")

## 💰 Cost: four levers that keep a run affordable

$$\text{cost per report} \approx \sum \text{calls} \times \text{tokens per call} \times \text{price per token}$$

| Lever | In our code |
|---|---|
| 1. **Model tiering** | `SMALL` for screener + reader (many calls), `STRONG` for planner, writer, critic |
| 2. **Fewer tokens** | abstracts cut to 600 chars, tool output cut to 4000 chars |
| 3. **Caching** | saved notes; reuse results for repeated queries |
| 4. **Hard limits** | `k=12` papers, max 2 revisions, token budget |

In [ ]:
PRICE_PER_MILLION_TOKENS_INR = 50     # 👈 put your provider's current price here (free tier = 0)
print(f"This notebook so far: {USAGE['calls']} LLM calls · {USAGE['tokens']:,} tokens "
      f"≈ ₹{USAGE['tokens'] / 1e6 * PRICE_PER_MILLION_TOKENS_INR:.2f}")

---
# Deploy: from notebook to a tool your department uses

```
Web UI (Streamlit) → API (FastAPI + login) → Orchestrator (graph + workers) → Tools (MCP servers)
Storage: SQLite checkpoints, Chroma notes · Tracing: Langfuse / LangSmith · Secrets: env vars, never in code
```
An 8-line Streamlit app is enough for a department pilot on one machine (`streamlit run app.py`):
```python
import streamlit as st
st.title("Research Assistant")
q = st.text_input("Your research question")
if st.button("Run") and q:
    with st.spinner("Researching…"):
        result = research(q)
    st.markdown(final_report(result))
```

---
# ⚖️ Responsible use: the assistant drafts, the scholar decides

| | |
|---|---|
| **Read before citing** | a summary is a pointer, not a substitute for the paper |
| **Disclose AI use** | follow your institution's and the journal's policy |
| **Protect data** | no unpublished or student data to external APIs (use a local model) |
| **Watch coverage bias** | English, open-access and preprint sources dominate |
| **Keep the audit trail** | the saved state shows how each paper was chosen |
| **Teach, don't replace** | students still learn to search, read and judge |

---
# 🏁 Capstone (20 min · teams of 2–3)
1. Pick a **real question** from your field → put it in `research("...")`
2. Run it and **approve the shortlist**
3. **Check three citations by hand** (non-negotiable)
4. Score it with the rubric: citation validity **30** · coverage **25** · faithfulness **25** · clarity/usefulness **20**
5. Share **one surprise** in 1 minute

### Take it further: student project ideas
Full-text PDF reader (GROBID) · citation graph · PRISMA-style systematic review · multilingual (Indian languages) · thesis "missing related work" reviewer · domain variants (PubMed, legal, patents)

### Further reading
Yao et al. 2022 *ReAct* · Shinn et al. 2023 *Reflexion* · Schick et al. 2023 *Toolformer* · Sumers et al. 2023 *CoALA* · Anthropic 2024 *Building Effective Agents* · Weng 2023 *LLM Powered Autonomous Agents* · modelcontextprotocol.io · a2a-protocol.org

> **Start simple, add autonomy where it pays, and keep a human in the loop.**